# Static linear encoding and Information Imbalance

This notebook combines bidirectional static linear encoding with Information Imbalance (II) at one neural-response timepoint. It performs the following operations:

1. Predict model features from the selected neural population response (`neural -> model`).
2. Predict the neural population response from model features (`model -> neural`).
3. Compute per-output $R^2$ for both projections.
4. Compute static II between the **predicted neural** and **predicted model** spaces.

By default, the projections are fitted and evaluated on all images (`use_cv=False`). Setting `use_cv=True` produces out-of-fold predictions, and II is then computed only from those out-of-fold predictions.

In [1]:
import os
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml
from torchvision.datasets import ImageFolder
from sklearn.decomposition import PCA

# Locate the repository whether Jupyter starts in the project root or scripts folder.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "config.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate config.yaml from the current directory.")
# end if PROJECT_ROOT is None

ENV = os.getenv("MY_ENV", "tiziano_mac_mini")
with open(PROJECT_ROOT / "config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
sys.path.append(paths["useful_stuff_path"])

# from II_analyses.static_encoding import (
#     compute_static_prediction_II,
#     fit_static_projection,
# )
from project_specific_utils.dataloader import (
    load_img_natraster,
    map_image_order_from_ann_to_monkey,
)
from useful_stuff.general_utils.regression import linear_encoding


In [13]:
@dataclass
class Cfg:
    # Neural recording and image-set parameters.
    monkey_name: str = "three0"
    date: str = "250313"
    brain_area: str = "AIT"
    folder_name: str = "talia_20each_tizi"
    new_fs: int = 100
    neural_timepoint_idx: int = 17
    pca_comp = 300
    # Static model-feature parameters.
    model_name: str = "dino_v3_l"
    img_size: int = 224
    layer_name: str = "layer.16.mlp.down_proj"
    pooling: str = "mean"

    # Linear encoding parameters. CV is deliberately disabled for this run.
    regression_type: str = "lr"
    use_cv: bool = False
    cv_type: str = "same"
    n_splits: int = 5
    shuffle: bool = True
    random_seed: int = 0
    alpha_min: float = 1e-6
    alpha_max: float = 1e3
    n_alphas: int = 10

    # Static Information Imbalance parameters.
    neural_RDM_metric: str = "cosine_cnt"
    model_RDM_metric: str = "cosine_cnt"
    k: int = 10
# EOC


cfg = Cfg()
cfg


Cfg(monkey_name='three0', date='250313', brain_area='AIT', folder_name='talia_20each_tizi', new_fs=100, neural_timepoint_idx=17, model_name='dino_v3_l', img_size=224, layer_name='layer.16.mlp.down_proj', pooling='mean', regression_type='lr', use_cv=False, cv_type='same', n_splits=5, shuffle=True, random_seed=0, alpha_min=1e-06, alpha_max=1000.0, n_alphas=10, neural_RDM_metric='cosine_cnt', model_RDM_metric='cosine_cnt', k=10)

## Load and align the static spaces

The neural raster is shaped `(neurons, timepoints, images)`. Model features are reordered to the monkey image-presentation order and remain shaped `(model features, images)`.

In [14]:
raster = load_img_natraster(
    paths,
    cfg.monkey_name,
    cfg.date,
    new_fs=cfg.new_fs,
    brain_area=cfg.brain_area,
)

dataset_path = Path(paths["livingstone_lab"]) / "Stimuli" / cfg.folder_name
dataset = ImageFolder(
    root=dataset_path,
    is_valid_file=lambda path: not path.endswith("Thumbs.db"),
    allow_empty=True,
)
idx_ord = map_image_order_from_ann_to_monkey(
    paths, cfg.monkey_name, cfg.date, dataset
)

features_path = (
    Path(paths["data_path"])
    / "models"
    / (
        f"{cfg.folder_name}_{cfg.model_name}_{cfg.img_size}_{cfg.layer_name}"
        f"_features_{cfg.pooling}pool.npz"
    )
)
model_features = np.load(features_path)["arr_0"][:, idx_ord]
if cfg.pca_comp is not None:
    pca_obj = PCA(n_components=cfg.pca_comp)
    model_features = pca_obj.fit_transform(model_features.T)
    model_features = model_features.T

    
raster_array = raster.get_array()

if raster_array.ndim != 3:
    raise ValueError(
        "Expected neural data with shape (neurons, timepoints, images), "
        f"received {raster_array.shape}."
    )
# end if raster_array.ndim != 3
if model_features.ndim != 2:
    raise ValueError(
        "Expected model data with shape (features, images), "
        f"received {model_features.shape}."
    )
# end if model_features.ndim != 2
if raster_array.shape[2] != model_features.shape[1]:
    raise ValueError(
        "Neural and model spaces have different image counts: "
        f"{raster_array.shape[2]} and {model_features.shape[1]}."
    )
# end if raster_array.shape[2] != model_features.shape[1]

print(f"Neural raster: {raster_array.shape}")
print(f"Aligned model features: {model_features.shape}")
print(f"Model features loaded from: {features_path}")


Neural raster: (25, 30, 776)
Aligned model features: (300, 776)
Model features loaded from: /Users/tizianocausin/IT_recap_local/models/talia_20each_tizi_dino_v3_l_224_layer.16.mlp.down_proj_features_meanpool.npz


In [ ]:
from useful_stuff.image_processing.computational_models import imgANN
ann = imgANN(
    model_name=cfg.model_name,
    pkg=cfg.pkg,
    img_size=cfg.img_size,
    pooling=cfg.pooling,
    weights_type=cfg.weights_type,
    dtype=dtype,
    attn_implementation=cfg.attn_implementation,
    repo_url=model_source,
    revision=cfg.revision,
    trust_remote_code=cfg.trust_remote_code,
    device=device,
)


# TODO 
1. import and load imgANN
2. make the forward pass work and comment that tidily
3. make the loss function (linear mapping and R^2)
4. make the training loop

In [ ]:
import torch
from torch import nn
import math
class BaselineModel(torch.nn.Module):
    def __init__(
            self, 
            encoder, 
            layers, 
            temporal_embedding_dim, 
            value_dim, 
            n_timepoints, 
            temporal_compression_ratio, 
            key_query_dim=None
            ):    
        super(BaselineModel, self).__init__()
        self.n_layers = len(layers)

        # set up the encoder
        self.encoder = encoder
        self.encoder.set_relevant_layers(layers)
        self.encoder_dim = self.encoder.get_layer_output_shape(layers[0])[2] # assumes all layers to have the same embedding dimension (Visual transformer case)

        # set up temporal embeddings
        # Learned coarse temporal queries:
        # (T_te, E_te)
        self.n_temporal_embeddings = n_timepoints / temporal_compression_ratio
        self.n_timepoints = n_timepoints
        self.temporal_embeddings = nn.Parameter(
            torch.randn(
                n_timepoints / temporal_compression_ratio, # TODO see how to treat the cases in which it doesn't yield an int
                temporal_embedding_dim,
            )
            * 0.02
        ) # TODO check out how to initialize

        # set up attention
        if key_query_dim is not None:
            self.key_dim = key_query_dim
            self.query_dim = key_query_dim
        else:
            self.key_dim = temporal_embedding_dim
            self.query_dim = temporal_embedding_dim
        # end if key_query_dim is not None:
        self.value_dim = value_dim

        # Backbone layer features -> keys
        # (B, L, E_in) -> (B, L, E_k)
        self.key_projection = nn.Linear(
            self.encoder_dim,
            self.key_dim,
            bias=False,
        )
        # Backbone layer features -> values
        # (B, L, E_in) -> (B, L, E_v)
        self.value_projection = nn.Linear(
            self.encoder_dim,
            self.value_dim,
            bias=False,
        )

        # normalization of projected layer features
        self.key_norm = nn.LayerNorm(self.key_dim)
        self.value_norm = nn.LayerNorm(self.value_dim)

        # Depthwise temporal upsampling:
        # (B, E_v, T_te) -> (B, E_v, T)
        self.temporal_upsampler = nn.ConvTranspose1d(
            in_channels=value_dim,
            out_channels=value_dim,
            kernel_size=temporal_compression_ratio,
            stride=temporal_compression_ratio, # TODO also change it
            groups=value_dim,
            bias=False,
        )

        # # One linear mapping shared over all timepoints:
        # # (B, T, E_v) -> (B, T, n_neurons)
        # self.neural_readout = nn.Linear(
        #     value_dim,
        #     out_dim,
        # ) 
        
        
    def forward(self, x):
        # extract feats from img encoder 
        # (B H W C) -> (B E L) # E = embedding dimension , L = number of layers
        layer_features = self.encoder(x) # TODO add the other stuff

        # cross-attention
        # project layers into K V
        # K: (B E L) -> (B E_t L) # E_t = temporal_embedding_dim
        # V: (B E L) -> (B E_v L) # E_v = value_embedding_dim
        # compute attention
        # Q, K, V, E_t -> (B E_v T_te) # T_te = resolution of temporal embeddings

        # Layer features become keys and values
        keys = self.key_projection(layer_features)
        keys = self.key_norm(keys)
        # (B, L, E_k)

        values = self.value_projection(layer_features)
        values = self.value_norm(values)
        # (B, L, E_v)
        # Temporal embeddings become queries
        queries = self.query_projection(self.temporal_embeddings)
        # Add batch dimension without creating copies
        queries = queries.unsqueeze(0).expand(layer_features.shape[0], -1, -1) # TODO find a more elegant way to define B

        # Scaled dot-product attention
        attention_logits = torch.matmul(
            queries,
            keys.transpose(-1, -2),
        )
        # (B, T_te, L)

        attention_logits = attention_logits / math.sqrt(self.key_dim)

        attention_weights = torch.softmax(
            attention_logits,
            dim=-1,
        )
        # Softmax across layers L

        # Weighted combination of layer values
        coarse_latents = torch.matmul(
            attention_weights,
            values,
        )

        # ConvTranspose1d expects channels-first format
        coarse_latents = coarse_latents.transpose(1, 2)
        # (B, E_v, T_te)

        fine_latents = self.temporal_upsampler(coarse_latents)
        # (B, E_v, T)

        fine_latents = fine_latents.transpose(1, 2)
        # (B, T, E_v)

        return fine_latents, attention_weights
        # (B, T_te, E_v)
        # (B, T_te, E_k)
        # (T_te, E_k)
        # Cross-attention
        #
        # attention_logits = Q @ K.transpose(-1, -2)
        #
        # attention_logits:
        # (B, T_te, E_k) @ (B, E_k, L)
        # -> (B, T_te, L)
        #
        # attention_weights = softmax(attention_logits / sqrt(E_k), dim=-1)
        #
        # attended_values = attention_weights @ V
        #
        # (B, T_te, L) @ (B, L, E_v)
        # -> (B, T_te, E_v)


        # upsampling (transpose_conv1d)
        # (B E_v T_te) -> (B E_v T)

        # sg(linear mapping)
        # (B E_v T) -> (B d T) # d = number of neurons
        
# 1. Extract features from frozen image encoder
# one pooled feature vector per selected layer
#
# raw features:
# (B, L, E_in)
#
# B    = batch size
# L    = number of selected layers, e.g. 3
# E_in = original backbone embedding dimension


# 2. Project layer features into keys and values
#
# K = key_projection(layer_features)
# V = value_projection(layer_features)
#
# K: (B, L, E_k)
# V: (B, L, E_v)


# 3. Create/project temporal embeddings into queries
#
# temporal_embeddings: (T_te, E_te)
# Q = query_projection(temporal_embeddings)
#
# Q: (T_te, E_k)
#
# expand over batch:
# Q: (B, T_te, E_k)





# 5. Learned temporal upsampling inside the network
#
# (B, T_te, E_v)
# -> (B, T, E_v)
#
# T_te = number of coarse temporal embeddings
# T    = number of neural timepoints, e.g. 30


# 6. One shared linear neural mapping
#
# (B, T, E_v)
# -> (B, T, d)
#
# d = number of neurons